In [1]:
import pandas as pd
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
orders = client.shop.orders

orders.insert_one({"_id": 1004, "customer": {"name": "Hung", "city": "Hue"},
                   "status": "paid", "items": []})

for doc in orders.find({"status": "paid"}, {"customer.name": 1, "status": 1}):
    print(doc)

pipeline = [
    {"$unwind": "$items"},
    {"$group": {"_id": "$items.sku", "units": {"$sum": "$items.qty"}}},
]
df = pd.DataFrame(orders.aggregate(pipeline))      # straight into pandas
print(df)

{'_id': 1001, 'customer': {'name': 'Linh'}, 'status': 'paid'}
{'_id': 1003, 'customer': {'name': 'Mai'}, 'status': 'paid'}
{'_id': 1004, 'customer': {'name': 'Hung'}, 'status': 'paid'}
     _id  units
0  BK-03      1
1  BK-01      3
2  BK-07      1


In [2]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client.shop
orders = db.orders

# Làm sạch collection trước khi test
orders.drop()

# 5 đơn hàng với shape khác nhau
sample_orders = [
    {
        "_id": 101,
        "customer": {"name": "An", "city": "Hanoi"},
        "status": "paid",
        "items": [
            {"sku": "SKU_A", "qty": 2, "price": 100},
            {"sku": "SKU_B", "qty": 1, "price": 50},
        ],
    },
    {
        "_id": 102,
        "customer": {"name": "Binh", "city": "Danang", "district": "Hai Chau"},
        "status": "pending",
        "items": [{"sku": "SKU_C", "qty": 5, "price": 20}],
        "discount_code": "SUMMER10",
    },
    {
        "_id": 103,
        "customer": {"name": "Chi", "city": "Hanoi"},
        "status": "paid",
        "items": [{"sku": "SKU_A", "qty": 1, "price": 100}],
        "delivery_notes": "Call before arrival",
    },
    {
        "_id": 104,
        "customer": {"name": "Dung", "city": "Hue"},
        "status": "paid",
        "items": [],  # Đơn hàng rỗng
        "is_gift": True,
    },
    {
        "_id": 105,
        "guest_checkout": True,
        "city": "Saigon",  # city đặt ở root thay vì lồng trong customer
        "items": [
            {"sku": "SKU_A", "qty": 3, "price": 100},
            {"sku": "SKU_D", "qty": 1, "price": 300},
        ],
    },
]

orders.insert_many(sample_orders)

# Tìm tất cả đơn hàng chứa SKU_A
target_sku = "SKU_A"
matching_orders = list(orders.find({"items.sku": target_sku}))

print(f"Số đơn chứa {target_sku}: {len(matching_orders)}")
for doc in matching_orders:
    print(f"- ID: {doc['_id']} | Customer: {doc.get('customer', doc.get('guest_checkout'))}")

Số đơn chứa SKU_A: 3
- ID: 101 | Customer: {'name': 'An', 'city': 'Hanoi'}
- ID: 103 | Customer: {'name': 'Chi', 'city': 'Hanoi'}
- ID: 105 | Customer: True


In [3]:
import pandas as pd

pipeline = [
    # 1. Tách từng item ra thành document riêng
    {"$unwind": "$items"},
    # 2. Nhóm theo city và tính tổng tiền
    {
        "$group": {
            "_id": "$customer.city",
            "total_revenue": {
                "$sum": {"$multiply": ["$items.qty", "$items.price"]}
            },
            "total_units": {"$sum": "$items.qty"},
        }
    },
    # 3. Sắp xếp doanh thu giảm dần
    {"$sort": {"total_revenue": -1}},
]

# Chuyển kết quả trực tiếp sang Pandas DataFrame để hiển thị sạch sẽ
df = pd.DataFrame(orders.aggregate(pipeline))
df.rename(columns={"_id": "city"}, inplace=True)
print("\n--- Revenue per City ---")
print(df)


--- Revenue per City ---
     city  total_revenue  total_units
0    None            600            4
1   Hanoi            350            4
2  Danang            100            5


In [4]:
import random

# Dùng collection riêng để test hiệu năng
test_col = db.perf_test
test_col.drop()

# Sinh 10.000 documents mẫu
print("Đang sinh 10,000 documents...")
cities = ["Hanoi", "Danang", "Hue", "Saigon", "Cantho"]
statuses = ["paid", "pending", "cancelled"]

docs = [
    {
        "order_code": f"ORD_{i}",
        "customer_id": random.randint(1, 1000),
        "city": random.choice(cities),
        "status": random.choice(statuses),
        "amount": round(random.uniform(10.0, 500.0), 2),
    }
    for i in range(10_000)
]
test_col.insert_many(docs)

# Truy vấn cần kiểm tra
query = {"customer_id": 425}

# 1. TRƯỚC KHI TẠO INDEX
explain_before = test_col.find(query).explain()
stats_before = explain_before["executionStats"]

# 2. TẠO INDEX
test_col.create_index([("customer_id", 1)])

# 3. SAU KHI TẠO INDEX
explain_after = test_col.find(query).explain()
stats_after = explain_after["executionStats"]

# In bảng so sánh số liệu thực tế
print("\n" + "=" * 50)
print(f"{'Chỉ số':<30} | {'Trước (No Index)':<16} | {'Sau (Indexed)':<16}")
print("-" * 50)

stage_before = stats_before["executionStages"]["stage"]
# Nếu có index, root stage thường là FETCH, stage con là IXSCAN
stage_after = stats_after["executionStages"]["stage"]

print(f"{'Execution Stage':<30} | {stage_before:<16} | {stage_after:<16}")
print(f"{'Total Docs Examined':<30} | {stats_before['totalDocsExamined']:<16} | {stats_after['totalDocsExamined']:<16}")
print(f"{'Total Keys Examined':<30} | {stats_before['totalKeysExamined']:<16} | {stats_after['totalKeysExamined']:<16}")
print(f"{'Execution Time (ms)':<30} | {stats_before['executionTimeMillis']:<16} | {stats_after['executionTimeMillis']:<16}")
print("=" * 50)

Đang sinh 10,000 documents...

Chỉ số                         | Trước (No Index) | Sau (Indexed)   
--------------------------------------------------
Execution Stage                | COLLSCAN         | FETCH           
Total Docs Examined            | 10000            | 12              
Total Keys Examined            | 0                | 12              
Execution Time (ms)            | 4                | 1               
